# EM Triage System Vector Database Creation

## Overview
This notebook creates vector databases for an Emergency Medicine (EM) triage system by processing two types of data:
1. **Medical Case Data**: Historical emergency cases with clinical summaries
2. **Triage Guidelines**: Emergency triage protocol sections with retrieval summaries

## Challenge 
Clinical notes are noisy and scattered, while triage guidelines are verbose and difficult to retrieve effectively. Direct embedding of raw data leads to poor semantic retrieval performance.

## Key Innovation
The approach uses specialized LLM agents to generate:
- **Clinical summaries** for medical cases optimized for semantic similarity matching
- **Retrieval summaries** for triage guideline sections optimized for case-to-guideline matching

These summaries are then embedded into vector databases, enabling more effective semantic retrieval compared to embedding raw data.

## Process Flow

### Medical Case Processing
1. Load medical case data from JSON files
2. Extract key metadata (demographics, vitals, disposition)
3. Generate clinical summaries using specialized medical LLM prompts
4. Create embeddings of the summaries
5. Store in vector database for similarity retrieval

### Triage Guideline Processing  
1. Load triage guideline sections from JSON files
2. Generate retrieval-optimized summaries for each section
3. Add summaries to guideline data for downstream use

## Supported LLMs
- **Claude 3.5 Sonnet** (Anthropic)
- **GPT-4o** (Azure OpenAI)
- **DeepSeek-V3** (DeepSeek API)

## Output
- Progressive vector databases (1000, 2000, 3000 cases)
- Enriched JSON files with LLM-generated summaries
- Processed triage guidelines with retrieval summaries

In [ ]:
import os
import json
import glob
import random
import shutil
import time
import uuid
from typing import Dict, List, Any, Optional
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# For loading environment variables
from dotenv import load_dotenv

## Initialize Embeddings and LLM 

This section initializes two key components:
1. **Azure OpenAI Embeddings**: Used to convert text summaries into vector representations
2. **LLM Model** (choose one): Used to generate clinical summaries from structured case data
    1. **DeepSeek-v3**
    2. **gpt-4o**
    3. **claude-3-7-sonnet**

Both components require API keys which are loaded from environment variables.

In [ ]:
# For loading environment variables
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify environment variables
required_env_vars = [
    "AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "DEEPSEEK_API_KEY",
    "AZURE_OPENAI_LLM_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
    "ANTHROPIC_API_KEY",
    "ANTHROPIC_MODEL_NAME",
]

missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing environment variables: {', '.join(missing_vars)}")

print("Environment variables loaded successfully.")

In [ ]:
def initialize_embeddings() -> AzureOpenAIEmbeddings:
    """Initialize Azure OpenAI embeddings"""
    return AzureOpenAIEmbeddings(
        azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    )


# Initialize embeddings
embeddings = initialize_embeddings()
print("Embeddings initialized successfully.")

In [ ]:
def initialize_deepseek(
    api_key: str = os.getenv("DEEPSEEK_API_KEY"),
    base_url: str = "https://api.deepseek.com",
    model_name: str = "deepseek-chat",
    temperature: float = 0,
):
    """
    Initialize DeepSeek client using LangChain

    Args:
        api_key (str): DeepSeek API key
        base_url (str): DeepSeek API base URL
        model_name (str): Model name to use
        temperature (float): Temperature for generation

    Returns:
        ChatOpenAI: Initialized DeepSeek client
    """
    return ChatOpenAI(
        model=model_name, api_key=api_key, base_url=base_url, temperature=temperature
    )


llm = initialize_deepseek()

print("LLM initialized successfully.")

model = "deepseekv3"

In [ ]:
from langchain_openai import AzureChatOpenAI


def initialize_azure_openai(
    azure_deployment_name: str = os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME"),
    azure_endpoint: str = os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_api_key: str = os.getenv("AZURE_OPENAI_API_KEY"),
    azure_api_version: str = os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature: float = 0,
):
    """
    Initialize Azure OpenAI client

    Args:
        azure_deployment_name (str): Azure deployment name
        azure_endpoint (str): Azure endpoint URL
        azure_api_key (str): Azure API key
        azure_api_version (str): Azure API version
        temperature (float): Temperature for generation

    Returns:
        AzureChatOpenAI: Initialized Azure OpenAI client
    """
    return AzureChatOpenAI(
        azure_deployment=azure_deployment_name,
        azure_endpoint=azure_endpoint,
        api_key=azure_api_key,
        api_version=azure_api_version,
        temperature=temperature,
    )


llm = initialize_azure_openai()

print("LLM initialized successfully.")

model = "gpt-4o"

In [ ]:
from langchain_anthropic import ChatAnthropic



def initialize_claude(

    api_key: str = os.getenv("ANTHROPIC_API_KEY"),

    model_name: str = os.getenv("ANTHROPIC_MODEL_NAME"),

    temperature: float = 0,

    max_tokens: int = 8192,

    streaming: bool = True,

):
    """

    Initialize Claude client using LangChain


    Args:

        api_key (str): Anthropic API key

        model_name (str): Model name to use (claude-3-7-sonnet-20250219 for latest Claude 3.7 Sonnet)

        temperature (float): Temperature for generation


    Returns:

        ChatAnthropic: Initialized Claude client
    """

    return ChatAnthropic(

        model=model_name,

        anthropic_api_key=api_key,

        temperature=temperature,

        max_tokens=max_tokens,

        streaming=streaming,

    )



llm = initialize_claude()


print("Claude LLM initialized successfully.")


model = "claude-3-7"

### =============================================================================
### Triage Guideline Processing 
### =============================================================================

### Triage Guideline Processing

This section processes triage guideline sections by adding LLM-generated summaries. The process involves:
1. Loading triage guideline sections from JSON
2. Generating summaries for each section using the selected LLM
3. Saving the enriched sections with summaries

This step is typically run once to prepare the triage guidelines for use in the RAG system.

In [ ]:
def load_sections_from_json(input_file: str) -> List[Dict[str, Any]]:
    """
    Load sections from a JSON file

    Args:
        input_file (str): Path to input JSON file

    Returns:
        List[Dict[str, Any]]: List of section data
    """
    with open(input_file, "r", encoding="utf-8") as f:
        sections = json.load(f)

    print(f"Loaded {len(sections)} sections from {input_file}")
    return sections


def generate_section_summary(section_title: str, section_content: str, llm) -> str:
    """
    Generate a summary for a section using an LLM

    Args:
        section_title (str): Title of the section
        section_content (str): Content of the section
        llm: LLM instance

    Returns:
        str: Summary of the section
    """
    # Prepare the prompt
    prompt = f"""
You are an expert emergency medicine triage nurse with extensive experience in rapid patient assessment and categorization.
TASK: Create a concise retrieval-optimized summary for the following emergency triage guideline section titled "{section_title}".
TRIAGE SECTION CONTENT:
{section_content}
INSTRUCTIONS:

Analyze the FULL SPECTRUM of clinical presentations covered in this section, from minimum threshold to most severe.
Identify the complete range of symptoms, vital signs, and conditions that would make this section relevant.
Create a summary that would help a retrieval system match this section to relevant patient cases across all severity levels.

YOUR SUMMARY SHOULD:

Begin with "Consult for patients with..."
Describe the SPECTRUM from minimum threshold conditions to most severe presentations
Format as: "Consult for patients with [condition spectrum] ranging from [MINIMUM threshold] to [most severe condition]"
Mention specific patient populations if applicable (pediatric, pregnant, etc.)
Be concise yet comprehensive (20-30 words maximum)
Use medical terminology that would appear in patient case descriptions
"""

    try:
        # Generate text using the provided LLM
        response = llm.invoke(prompt)

        # Extract the content from the response
        if hasattr(response, "content"):  # Azure/OpenAI/Anthropic style
            summary = response.content.strip()
        else:  # Ollama style
            summary = str(response).strip()

        return summary

    except Exception as e:
        print(f"Error generating summary: {str(e)}")
        return "Failed to generate summary due to error."


def process_all_sections(sections: List[Dict[str, Any]], output_file: str, llm):
    """
    Process all sections, generate summaries, and save results

    Args:
        sections (List[Dict[str, Any]]): List of section data
        output_file (str): Path to output JSON file
        llm: LLM instance to use
    """
    total_sections = len(sections)
    print(f"Starting to process {total_sections} sections...")

    for i, section in enumerate(sections, 1):
        section_title = section["section_title"]
        section_content = section["content"]

        print(f"Processing section {i}/{total_sections}: {section_title}")

        # Generate summary using the provided LLM instance
        summary = generate_section_summary(section_title, section_content, llm)

        # Add summary to section data
        section["summary"] = summary

        print(f"  Summary: {summary}")

        # Add a small delay to not overload the server
        if i < total_sections:
            time.sleep(0.5)

    # Save the updated sections
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(sections, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(sections)} sections with summaries to {output_file}")

In [ ]:
def process_triage_guidelines(
    sections_json_path: str, output_json_path: str, db_output_path: str
):
    """
    Process triage guidelines by adding LLM summaries

    Args:
        sections_json_path (str): Path to input sections JSON
        output_json_path (str): Path to save output with summaries
        db_output_path (str): Path to save copy for database
    """
    print("=== Processing Triage Guidelines ===")

    # Load sections
    sections = load_sections_from_json(sections_json_path)

    # Process sections
    process_all_sections(sections, output_json_path, llm)

    # Create db directory if it doesn't exist and save copy
    os.makedirs(os.path.dirname(db_output_path), exist_ok=True)
    with open(output_json_path, "r", encoding="utf-8") as f:
        sections_with_summaries = json.load(f)

    with open(db_output_path, "w", encoding="utf-8") as f:
        json.dump(sections_with_summaries, f, indent=2, ensure_ascii=False)

    print(f"Saved additional copy to {db_output_path}")
    print("Triage guideline processing complete!")

In [ ]:
process_triage_guidelines(
    sections_json_path="../data/guideline/triage_guideline_sections.json",
    output_json_path=f"../data/guideline/triage_sections_with_summaries_{model}.json",
    db_output_path=f"../db/triage_sections_with_summaries_{model}.json",
)

### =============================================================================
### Past Case Processing 
### =============================================================================

## Define Functions

The following functions implement the core processing pipeline:
1. `create_directory_structure`: Create directory structure for vector databases and JSON storage
2. `get_category_files`: Get files from each category folder
3. `sample_files_proportionally`: Sample files proportionally from each category.
4. `load_sampled_json_files`: Load JSON content from the sampled file paths.
5. `extract_metadata`: Extract metadata from the JSON data according to the requirements.
6. `process_case_json`: Process each json case into clinical summary
7. `prepare_documents_for_vectordb`: Prepare documents for vector databases and store enriched JSON files. Skip files that have already been processed.
8. `create_vector_databases`: Create a vector database, using existing data if available.

In [ ]:
def create_directory_structure(base_dir: str) -> Dict[str, str]:
    """
    Create directory structure for vector databases and JSON storage

    Args:
        base_dir: The base directory for all storage

    Returns:
        A dictionary of directory paths
    """
    # Create main directories
    dirs = {
        "base": base_dir,
        "json_store": os.path.join(base_dir, "json_store"),
        "summary_vectordb": os.path.join(base_dir, "summary_vectordb"),
    }

    # Create directories if they don't exist
    for dir_path in dirs.values():
        os.makedirs(dir_path, exist_ok=True)
        print(f"Created directory: {dir_path}")

    return dirs


def get_category_files(directory: str) -> Dict[str, List[str]]:
    """
    Get files from each category folder.

    Args:
        directory: Base directory containing category folders

    Returns:
        Dictionary with category names as keys and lists of file paths as values
    """
    category_files = {}

    # Find all category folders
    category_dirs = glob.glob(os.path.join(directory, "Cat *"))

    for category_dir in category_dirs:
        category_name = os.path.basename(category_dir)
        file_paths = glob.glob(os.path.join(category_dir, "*.json"))
        category_files[category_name] = file_paths

    return category_files


def sample_files_proportionally(
    category_files: Dict[str, List[str]],
    total_sample_size: int,
    remove_from_original: bool = True,
) -> List[str]:
    """
    Sample files proportionally from each category.

    Args:
        category_files: Dictionary with category names as keys and lists of file paths as values
        total_sample_size: Total number of files to sample
        remove_from_original: Whether to remove sampled files from the original lists

    Returns:
        List of file paths that have been sampled
    """
    # Copy the dictionary if we don't want to modify the original
    if not remove_from_original:
        category_files = {k: v.copy() for k, v in category_files.items()}

    # Count total files
    total_files = sum(len(files) for files in category_files.values())

    # Calculate proportional sample size for each category
    sampled_files = []

    # Initial calculation of proportional samples
    sample_sizes = {}
    for category, files in category_files.items():
        if not files:  # Skip empty categories
            sample_sizes[category] = 0
            continue

        proportion = len(files) / total_files
        sample_size = max(1, int(round(proportion * total_sample_size)))
        sample_sizes[category] = min(sample_size, len(files))

    # Adjust to match total_sample_size exactly
    total_allocated = sum(sample_sizes.values())

    if total_allocated != total_sample_size:
        # Get categories that can be adjusted (have more files than currently sampled)
        adjustable_categories = [
            cat
            for cat in category_files.keys()
            if len(category_files[cat]) > sample_sizes[cat]
            or (sample_sizes[cat] > 1 and total_allocated > total_sample_size)
        ]

        if adjustable_categories:
            if total_allocated < total_sample_size:
                # Need to add more files
                add_count = total_sample_size - total_allocated
                for _ in range(add_count):
                    # Choose a category that still has files available
                    available_cats = [
                        cat
                        for cat in adjustable_categories
                        if len(category_files[cat]) > sample_sizes[cat]
                    ]
                    if not available_cats:
                        break
                    category = random.choice(available_cats)
                    sample_sizes[category] += 1
            else:
                # Need to remove some files
                remove_count = total_allocated - total_sample_size
                for _ in range(remove_count):
                    # Choose a category that has more than 1 file allocated
                    reducible_cats = [
                        cat for cat in adjustable_categories if sample_sizes[cat] > 1
                    ]
                    if not reducible_cats:
                        break
                    # Choose categories with more files first
                    category = max(reducible_cats, key=lambda cat: sample_sizes[cat])
                    sample_sizes[category] -= 1

    # Now perform the actual sampling
    for category, sample_size in sample_sizes.items():
        if sample_size <= 0 or not category_files[category]:
            continue

        category_sample = random.sample(category_files[category], sample_size)
        sampled_files.extend(category_sample)

        # Remove sampled files from the original list if specified
        if remove_from_original:
            for file_path in category_sample:
                category_files[category].remove(file_path)

    # Print sampling results
    print(f"Sampled {len(sampled_files)} files in total")
    category_distribution = Counter(
        [os.path.dirname(file).split(os.sep)[-1] for file in sampled_files]
    )
    for category, count in category_distribution.items():
        print(f"  {category}: {count} files")

    return sampled_files


def load_sampled_json_files(file_paths: List[str]) -> List[Dict[str, Any]]:
    """
    Load JSON content from the sampled file paths.

    Args:
        file_paths: List of file paths to load

    Returns:
        List of dictionaries with content and metadata
    """
    loaded_files = []

    for file_path in file_paths:
        try:
            with open(file_path, "r") as file:
                data = json.load(file)

            # Generate a unique UUID for the document
            doc_id = str(uuid.uuid4())

            # Add file metadata
            file_info = {
                "content": data,
                "original_filename": os.path.basename(file_path),
                "doc_id": doc_id,
            }
            loaded_files.append(file_info)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

    print(f"Successfully loaded {len(loaded_files)} JSON files")
    return loaded_files

In [ ]:
def extract_metadata(json_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extract metadata from the JSON data according to the requirements.

    Args:
        json_data: The JSON data from which to extract metadata

    Returns:
        A dictionary containing the extracted metadata
    """
    metadata = {}

    # Add everything under "Case Disposition"
    if "Case Disposition" in json_data:
        for key in [
            "Triage Category",
            "Diagnosis",
            "Attending Specialty",
            "Trauma Type",
            "Discharge Destination",
        ]:
            if key in json_data["Case Disposition"]:
                metadata[key] = json_data["Case Disposition"][key]

    # Add specific fields from "Demographics"
    if "Demographics" in json_data:
        for key in [
            "Age",
            "Sex",
            "Ambulatory status",
            "Informant",
            "TOCC",
            "Referral (if any)",
        ]:
            if key in json_data["Demographics"]:
                metadata[key] = json_data["Demographics"][key]

        # Add age_group classification based on Age
        if "Age" in json_data["Demographics"]:
            age_text = json_data["Demographics"]["Age"]

            # Default age_group if we can't parse the age
            metadata["age_group"] = "Unknown"

            # Check for years format (e.g., "87 years")
            if "years" in age_text.lower():
                try:
                    # Extract the numeric part
                    age_years = int(age_text.lower().split("years")[0].strip())
                    if age_years >= 18:
                        metadata["age_group"] = "Adult"
                    else:
                        metadata["age_group"] = "Paediatric"
                except (ValueError, IndexError):
                    # If parsing fails, leave as Unknown
                    pass

            # Check for months format (e.g., "9 months")
            elif "months" in age_text.lower() or "month" in age_text.lower():
                metadata["age_group"] = (
                    "Paediatric"  # All month-based ages are Paediatric
                )

    return metadata

In [ ]:
def process_case_json(case: Dict[str, Any], llm: ChatOpenAI) -> str:
    """
    Process a case JSON file to generate a clinical summary using the LLM.

    Args:
        case: The case data as a dictionary
        llm: The initialized LLM to use for generating the summary

    Returns:
        A string containing the generated clinical summary
    """
    # Step 1. Extract Sections for Summary
    sections_for_summary = {}
    if "Demographics" in case:
        if "Age" in case["Demographics"]:
            sections_for_summary["Age"] = case.get("Demographics", {}).get("Age", "")
        if "Sex" in case["Demographics"]:
            sections_for_summary["Sex"] = case.get("Demographics", {}).get("Sex", "")
    if "Vitals and Observations" in case:
        sections_for_summary["Vitals and Observations"] = case.get(
            "Vitals and Observations", {}
        )
    if "Clinical Presentation" in case:
        sections_for_summary["Clinical Presentation"] = case.get(
            "Clinical Presentation", {}
        )

    summary_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
        You are a specialized medical summarization assistant for emergency triage notes. Your task is to analyze structured JSON data from triage notes and create a concise, retrievable summary highlighting only the most important clinical information.
        ## INPUT STRUCTURE
        You will receive structured JSON data containing:
        - Demographics (sex, age)
        - Clinical Presentation (chief complaint, condition on arrival)
        - Vitals and Observations

        ## OUTPUT REQUIREMENTS
        1. Generate a concise (2-3 sentence) summary focusing on:
           - Patient demographics (sex, age)
           - Key presenting complaints
           - Abnormal physical findings
           - ONLY abnormal vital signs according to strict criteria

        2. Write in a factual, medical style optimized for vector embedding retrieval.

        ## MEDICAL TERMINOLOGY GUIDELINES
        - Interpret medical abbreviations contextually 
        - Fix obvious typos/misspellings based on medical context
        - Maintain medical precision while expanding clarity
        - For Hong Kong hospital/facility abbreviations:
          - "PMH" → "Princess Margaret Hospital"
          - "KCH" → "Kwai Chung Hospital"
          - "OAH" → "Old Age Home"
          - If uncertain about any abbreviation, do NOT expand it

        ## VITAL SIGNS ABNORMALITY CRITERIA
        ONLY mention vital signs that meet these specific abnormality thresholds:

        - Blood Pressure (Adult):
          - Hypertension: SBP > 180 mmHg or DBP > 100 mmHg
          - Hypotension: SBP < 90 mmHg

        - Heart Rate (Adults):
          - Tachycardia: HR > 120/min
          - Bradycardia: HR < 50/min

        - Temperature (Adult):
          - Fever: Temp ≥ 38°C
          - Hypothermia: Temp < 35°C

        - Blood Sugar:
          - Hyperglycemia: H'stix > 16 mmol/l
          - Hypoglycemia: H'stix < 3.9 mmol/l

        - Respiratory Status (always consider):
          - Oxygen supplementation (if present)
          - SpO2 level if abnormal
          - Abnormal respiratory rate

        - Conscious Level:
          - GCS score if not 15/15
          - Pupil abnormalities (if provided)

        - Pregnancy/Menstruation: Note if pregnant or abnormal menstrual patterns

        ## EXAMPLES OF GOOD SUMMARIES:

        ### Example 1:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "F",
                    "Age": "56 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "R/F ? IHD, tachycardia oedema\\nSOB , LL edema and neck swelling x 10/7 \\nchest discomfort - , palpitiation",
                    "Condition on Arrival": "speak in sentence"
                }},
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "E: 4, V: 5, M: 6, Score: 15/15",
                        "BP": "1st: 144/89 mmHg",
                        "PR/AR": "99",
                        "Temp": "36.2",
                        "SpO2": "100% on room air, --",
                        "RR": "--",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }},
                    "Menstruation / Pregnancy Status": "unspecified"
                }}
            }}
        ]
        
        Output: "56-year-old female presenting with shortness of breath, lower limb edema, and neck swelling for 10 days, accompanied by chest discomfort and palpitations. Patient can speak in full sentences and has been referred for evaluation of possible ischemic heart disease with tachycardia and edema. All vital signs are within normal ranges."

        ### Example 2:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "M",
                    "Age": "64 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "SOB this morning 0730\\nroom air 70%",
                    "Condition on Arrival": "no lower limb oedema\\nsputum sound+/-"
                }},    
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "",
                        "BP": "1st: 165/97 mmHg",
                        "PR/AR": "130",
                        "Temp": "38.1",
                        "SpO2": "96% on 6 L/min via Nasal Cannula",
                        "RR": "20",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }}
                }}
            }}
        ]
        Output: "64-year-old male presenting with sudden onset shortness of breath at 7:30 AM with significant hypoxia (70% on room air). Patient has tachycardia (HR 130), fever (38.1°C), and requires oxygen supplementation (6 L/min via Nasal Cannula). Clinical examination reveals sputum sounds without lower limb edema."

        ## IMPORTANT REMINDERS:
        - Focus on clinical relevance and retrieval effectiveness
        - Maintain medical accuracy while being concise
        - Only highlight abnormal vitals based on specific thresholds
        - Fix obvious errors but maintain meaning integrity
        - Adjust abnormality thresholds for pediatric cases
        """,
            ),
            (
                "user",
                """
        Case Data (JSON):
        {case_data}

        OUTPUT (Summary):
        """,
            ),
        ]
    )

    # Prepare the input for the prompt
    case_data_str = json.dumps(sections_for_summary, indent=2)
    input_values = {"case_data": case_data_str}
    # Get the LLM response
    chain = summary_prompt | llm | StrOutputParser()
    response = chain.invoke(input_values)

    print("============= Summary Created =============")
    print(response)

    return response

In [ ]:
def prepare_documents_for_vectordb(
    json_files: List[Dict[str, Any]], storage_dirs: Dict[str, str], llm: ChatOpenAI
) -> List[Document]:
    """
    Prepare documents for vector databases and store enriched JSON files.
    Skip files that have already been processed.

    Args:
        json_files: A list of dictionaries containing JSON content and file metadata.
        storage_dirs: A dictionary of directory paths, including 'json_store'.
        llm: The LLM to use for generating summaries

    Returns:
        A list of Document objects for use in vector databases.
    """
    json_store_dir = storage_dirs["json_store"]
    os.makedirs(json_store_dir, exist_ok=True)

    summary_docs = []
    total_files = len(json_files)
    processed_count = 0
    skipped_count = 0

    for idx, file_info in enumerate(json_files, start=1):
        filename = file_info["original_filename"]
        doc_id = file_info["doc_id"]

        # Check if file already exists in JSON store
        output_path = os.path.join(json_store_dir, f"{doc_id}.json")
        if os.path.exists(output_path):
            print(f"Skipping {idx}/{total_files}: {filename} (already processed)")

            # Load existing enriched JSON
            with open(output_path, "r") as f:
                enriched_json = json.load(f)

            # Extract metadata from original file for consistency
            metadata = extract_metadata(file_info["content"])
            metadata["doc_id"] = doc_id
            metadata["original_filename"] = filename

            # Create Document from existing summary
            summary_text = enriched_json.get("_summary_text", "")
            if summary_text:
                summary_doc = Document(page_content=summary_text, metadata=metadata)
                summary_docs.append(summary_doc)
                skipped_count += 1
                continue

        print("=" * 80)
        print(f"Processing {idx}/{total_files}: {filename}")

        json_data = file_info["content"]

        # Extract metadata
        metadata = extract_metadata(json_data)
        metadata["doc_id"] = doc_id
        metadata["original_filename"] = filename

        # Generate summary
        try:
            summary_text = process_case_json(json_data, llm)

            # Add summary to JSON before saving
            enriched_json = json_data.copy()
            enriched_json["_summary_text"] = summary_text

            # Save enriched JSON
            with open(output_path, "w") as f:
                json.dump(enriched_json, f, indent=2)

            # Create Document for vector DB
            summary_doc = Document(page_content=summary_text, metadata=metadata)
            summary_docs.append(summary_doc)
            processed_count += 1

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            # Continue with next file instead of stopping the entire process

    print(
        f"Processed {processed_count} new files, skipped {skipped_count} existing files"
    )
    print(f"Total of {len(summary_docs)} documents ready for vector database")
    return summary_docs

In [ ]:
def create_vector_databases(
    json_files: List[Dict[str, Any]],
    embeddings: AzureOpenAIEmbeddings,
    storage_dirs: Dict[str, str],
    llm: ChatOpenAI,
) -> Chroma:
    """
    Create a vector database, using existing data if available.

    Args:
        json_files: A list of dictionaries containing JSON content and file paths
        embeddings: The embeddings object to use
        storage_dirs: A dictionary of directory paths
        llm: The LLM to use for generating summaries

    Returns:
        A Chroma vector database object
    """
    # Check if the vector database already exists
    summary_vectordb_path = storage_dirs["summary_vectordb"]

    # Prepare documents for vector databases
    summary_docs = prepare_documents_for_vectordb(
        json_files=json_files, storage_dirs=storage_dirs, llm=llm
    )

    print(f"Prepared {len(summary_docs)} documents for Summary vector database")

    # Create a list of document IDs to use in databases
    doc_ids = [doc.metadata["doc_id"] for doc in summary_docs]

    # Check if there's an existing database - if yes, we'll just add new documents
    if os.path.exists(summary_vectordb_path) and os.listdir(summary_vectordb_path):
        print("Found existing vector database. Loading...")
        try:
            # First try to load existing database
            summary_vectordb = Chroma(
                embedding_function=embeddings,
                persist_directory=summary_vectordb_path,
                collection_name="summary_text",
            )

            # Get existing IDs to avoid duplicates
            existing_ids = set(summary_vectordb.get()["ids"])

            # Filter out documents that are already in the database
            new_docs = []
            new_ids = []
            for doc, doc_id in zip(summary_docs, doc_ids):
                if doc_id not in existing_ids:
                    new_docs.append(doc)
                    new_ids.append(doc_id)

            if new_docs:
                print(
                    f"Adding {len(new_docs)} new documents to existing vector database"
                )
                summary_vectordb.add_documents(new_docs, ids=new_ids)
            else:
                print("No new documents to add to vector database")

        except Exception as e:
            print(f"Error loading existing database: {str(e)}")
            print("Creating new vector database instead")
            summary_vectordb = Chroma.from_documents(
                documents=summary_docs,
                embedding=embeddings,
                persist_directory=summary_vectordb_path,
                collection_name="summary_text",
                collection_metadata={"hnsw:space": "cosine"},
                ids=doc_ids,
            )
    else:
        # Create new vector database if one doesn't exist
        print("Creating new vector database")
        summary_vectordb = Chroma.from_documents(
            documents=summary_docs,
            embedding=embeddings,
            persist_directory=summary_vectordb_path,
            collection_name="summary_text",
            collection_metadata={"hnsw:space": "cosine"},
            ids=doc_ids,
        )

    # Persist the vector database
    summary_vectordb.persist()

    print("Vector database updated and persisted successfully")

    return summary_vectordb

## Pipeline execution 
The following functions are for execution of the processing pipeline:
1. `run_until_complete_processing`: Run the vector database creation process until all files are processed without errors.
2. `progressive_vector_db_creation`: Create progressive vector databases with proportional representation from each category.
3. `validate_vector_database`: Validate the vector database by comparing summaries with their source JSON files and verifying UUID consistency between vector DB and JSON store.


In [ ]:
def run_until_complete_processing(
    json_files: List[Dict[str, Any]], storage_base_dir: str, max_attempts: int = 10
):
    """
    Run the vector database creation process until all files are processed without errors.

    Args:
        json_files: Pre-loaded list of JSON files with consistent IDs
        storage_base_dir: Base directory for storage
        max_attempts: Maximum number of attempts

    Returns:
        The final vector database
    """
    # Create directory structure
    storage_dirs = create_directory_structure(storage_base_dir)

    # Initialize embeddings and LLM
    embeddings = initialize_embeddings()
    #     llm = initialize_deepseek()
    llm = initialize_claude()

    attempt = 0
    unprocessed_files = True

    while unprocessed_files and attempt < max_attempts:
        attempt += 1
        print(f"\n{'='*30} ATTEMPT {attempt} OF {max_attempts} {'='*30}\n")

        # Count files in JSON store
        json_store_dir = storage_dirs["json_store"]
        before_count = (
            len(os.listdir(json_store_dir)) if os.path.exists(json_store_dir) else 0
        )

        # Create/update vector databases
        summary_vectordb = create_vector_databases(
            json_files=json_files,
            embeddings=embeddings,
            storage_dirs=storage_dirs,
            llm=llm,
        )

        # Check if all files were processed
        after_count = len(os.listdir(json_store_dir))
        total_files = len(json_files)

        print(f"\nProcessed files: {after_count}/{total_files}")
        print(f"New files in this attempt: {after_count - before_count}")

        # If we've processed all files or no new files were processed in this attempt, we're done
        if after_count >= total_files or (after_count - before_count) == 0:
            unprocessed_files = False
            print("\nAll files have been processed successfully!")
        else:
            print(
                f"\n{total_files - after_count} files still need processing. Waiting 5 seconds before next attempt..."
            )
            time.sleep(5)

    if attempt >= max_attempts and unprocessed_files:
        print(
            f"\nReached maximum attempts ({max_attempts}) with some files still unprocessed."
        )

    return summary_vectordb

In [ ]:
def progressive_vector_db_creation(
    data_directory: str,
    db_base_path: str,
    files_per_stage: int = 1000,
    max_files: int = 3000,
):
    """
    Create progressive vector databases with proportional representation from each category.
    FIXED version to avoid re-embedding previously processed documents.

    Args:
        data_directory: Directory containing category subfolders with JSON files
        db_base_path: Base path for creating database directories
        files_per_stage: Number of files to add in each stage
        max_files: Maximum number of files to process in total
    """
    # Get files from each category
    print(f"\n{'='*30} INITIALIZING PROGRESSIVE DATABASE CREATION {'='*30}")
    print(f"Data directory: {data_directory}")
    print(f"Files per stage: {files_per_stage}")
    print(f"Maximum files: {max_files}")

    # Get all files from each category
    category_files = get_category_files(data_directory)

    # Print initial category distribution
    total_available = sum(len(files) for files in category_files.values())
    print(f"\nInitial category distribution (Total: {total_available} files):")
    for category, files in category_files.items():
        print(
            f"  {category}: {len(files)} files ({len(files)/total_available*100:.1f}%)"
        )

    # Calculate number of stages
    num_stages = min(total_available, max_files) // files_per_stage
    if min(total_available, max_files) % files_per_stage > 0:
        num_stages += 1

    print(f"\nPlanned stages: {num_stages}")

    # Initialize embeddings for consistent usage across stages
    embeddings = initialize_embeddings()
    #     llm = initialize_deepseek()
    llm = initialize_claude()

    # Process each stage
    all_processed_files = []

    for stage in range(1, num_stages + 1):
        current_stage_size = files_per_stage
        current_total_size = stage * files_per_stage

        # Adjust for last stage if needed
        if current_total_size > min(total_available, max_files):
            current_stage_size = min(total_available, max_files) - (
                (stage - 1) * files_per_stage
            )

        print(
            f"\n{'='*30} STAGE {stage}/{num_stages}: Adding {current_stage_size} files {'='*30}"
        )

        # Sample files for this stage with proportional representation
        stage_files = sample_files_proportionally(category_files, current_stage_size)

        # Load the sampled JSON files
        stage_json_files = load_sampled_json_files(stage_files)

        # Create stage-specific database directory
        stage_db_path = f"{db_base_path}_{current_total_size}case"

        # Create directory structure for this stage
        storage_dirs = create_directory_structure(stage_db_path)

        # If not the first stage, copy the previous stage's database content
        previous_vectordb = None
        if stage > 1:
            previous_db_path = f"{db_base_path}_{(stage-1) * files_per_stage}case"

            if os.path.exists(previous_db_path):
                print(f"Copying content from previous stage: {previous_db_path}")

                # Copy JSON store directory
                prev_json_store = os.path.join(previous_db_path, "json_store")
                new_json_store = os.path.join(stage_db_path, "json_store")

                if os.path.exists(prev_json_store) and os.path.isdir(prev_json_store):
                    print(
                        f"Copying JSON store from previous stage: {prev_json_store} -> {new_json_store}"
                    )
                    if os.path.exists(new_json_store):
                        shutil.rmtree(new_json_store)
                    shutil.copytree(prev_json_store, new_json_store)

                # Copy vector database
                prev_vectordb_path = os.path.join(previous_db_path, "summary_vectordb")
                new_vectordb_path = os.path.join(stage_db_path, "summary_vectordb")

                if os.path.exists(prev_vectordb_path) and os.path.isdir(
                    prev_vectordb_path
                ):
                    print(
                        f"Copying vector database from previous stage: {prev_vectordb_path} -> {new_vectordb_path}"
                    )
                    if os.path.exists(new_vectordb_path):
                        shutil.rmtree(new_vectordb_path)
                    shutil.copytree(prev_vectordb_path, new_vectordb_path)

                    # Load the previous vector database
                    try:
                        previous_vectordb = Chroma(
                            embedding_function=embeddings,
                            persist_directory=new_vectordb_path,
                            collection_name="summary_text",
                        )
                        print(
                            f"Successfully loaded previous vector database with {len(previous_vectordb.get()['ids'])} documents"
                        )
                    except Exception as e:
                        print(f"Error loading previous vector database: {str(e)}")
                        previous_vectordb = None

        # Process only the NEW files for this stage
        print(f"\nProcessing {len(stage_json_files)} new files for stage {stage}")

        # Prepare documents for vector database from the new files only
        summary_docs = prepare_documents_for_vectordb(
            json_files=stage_json_files, storage_dirs=storage_dirs, llm=llm
        )

        print(f"Prepared {len(summary_docs)} documents for embedding")

        # Create document IDs
        doc_ids = [doc.metadata["doc_id"] for doc in summary_docs]

        # Now handle the vector database, using the previous one if available
        if previous_vectordb is not None:
            try:
                # Add new documents to the existing database
                if summary_docs:
                    print(
                        f"Adding {len(summary_docs)} new documents to the copied vector database"
                    )
                    previous_vectordb.add_documents(summary_docs, ids=doc_ids)
                    previous_vectordb.persist()
                print(
                    f"Updated vector database now contains {len(previous_vectordb.get()['ids'])} documents"
                )
                summary_vectordb = previous_vectordb
            except Exception as e:
                print(f"Error adding documents to copied vector database: {str(e)}")
                print("Creating new vector database with all documents as fallback")
                # Add current stage files to the cumulative list
                all_processed_files.extend(stage_json_files)
                # Recreate the vector database as a fallback
                summary_vectordb = create_vector_databases(
                    json_files=all_processed_files,
                    embeddings=embeddings,
                    storage_dirs=storage_dirs,
                    llm=llm,
                )
        else:
            # First stage or previous database not available
            print("Creating new vector database")
            # Add current stage files to the cumulative list
            all_processed_files.extend(stage_json_files)
            # Create a new vector database with the current files
            if stage == 1:
                # For first stage, just use stage_json_files
                summary_vectordb = create_vector_databases(
                    json_files=stage_json_files,
                    embeddings=embeddings,
                    storage_dirs=storage_dirs,
                    llm=llm,
                )
            else:
                # If we got here but it's not the first stage, something went wrong with copying
                # Use all processed files as a fallback
                summary_vectordb = create_vector_databases(
                    json_files=all_processed_files,
                    embeddings=embeddings,
                    storage_dirs=storage_dirs,
                    llm=llm,
                )

        print(f"\nCompleted Stage {stage}/{num_stages}")
        print(
            f"Database with {len(summary_vectordb.get()['ids'])} files saved at: {stage_db_path}"
        )

    print(f"\n{'='*30} ALL STAGES COMPLETED SUCCESSFULLY {'='*30}")
    print(f"Created {num_stages} progressive databases")

    # Return information about the completed process
    return {
        "total_stages": num_stages,
        "total_files_processed": len(all_processed_files),
        "final_database_path": f"{db_base_path}_{min(total_available, max_files)}case",
    }

In [ ]:
def validate_vector_database(
    vectordb: Chroma,
    json_store_dir: str,
    num_samples: int = 5,
    check_all_uuids: bool = True,
):
    """
    Validate the vector database by comparing summaries with their source JSON files
    and verifying UUID consistency between vector DB and JSON store.

    Args:
        vectordb: The Chroma vector database to validate
        json_store_dir: Directory containing the enriched JSON files
        num_samples: Number of random samples to check for content validation
        check_all_uuids: Whether to check UUID consistency for all documents, not just samples
    """
    print(f"\n{'='*30} DATABASE VALIDATION {'='*30}")

    # Get all entries from the vector database
    all_results = vectordb.get()
    all_ids = all_results["ids"]
    all_documents = all_results["documents"]
    all_metadatas = all_results["metadatas"]

    total_entries = len(all_ids)
    print(f"Total entries in vector database: {total_entries}")

    if total_entries == 0:
        print("Database is empty. Nothing to validate.")
        return False

    # PART 1: UUID VALIDATION - Check if all UUIDs in vector DB have corresponding JSON files
    if check_all_uuids:
        print(f"\n{'='*30} UUID VALIDATION {'='*30}")
        print(f"Checking UUID consistency for all {total_entries} entries...")

        # Get all JSON files in the store
        json_files = os.listdir(json_store_dir)
        json_uuids = set(
            [os.path.splitext(f)[0] for f in json_files if f.endswith(".json")]
        )

        # Get all vector DB UUIDs
        vector_uuids = set(all_ids)

        # Check for mismatches
        missing_in_json = vector_uuids - json_uuids
        missing_in_vector = json_uuids - vector_uuids

        uuid_validation_success = True

        if missing_in_json:
            print(
                f"❌ Found {len(missing_in_json)} UUIDs in vector DB that are missing in JSON store"
            )
            print(f"  First 5 examples: {list(missing_in_json)[:5]}")
            uuid_validation_success = False
        else:
            print(f"✅ All vector DB UUIDs have corresponding JSON files")

        if missing_in_vector:
            print(
                f"⚠️ Found {len(missing_in_vector)} UUIDs in JSON store that are missing in vector DB"
            )
            print(f"  First 5 examples: {list(missing_in_vector)[:5]}")
            # This is not necessarily an error - JSON store might contain files not yet added to vector DB
        else:
            print(f"✅ All JSON store UUIDs are present in the vector DB")

        print(
            f"UUID validation {'successful' if uuid_validation_success else 'failed'}"
        )

    # PART 2: CONTENT VALIDATION - Check content for random samples
    print(f"\n{'='*30} CONTENT VALIDATION {'='*30}")

    # Select random samples for validation
    import random

    sample_indices = random.sample(
        range(total_entries), min(num_samples, total_entries)
    )

    content_validation_successful = True

    for i, idx in enumerate(sample_indices, 1):
        doc_id = all_ids[idx]
        db_summary = all_documents[idx]
        db_metadata = all_metadatas[idx]

        print(f"\n{'*'*20} SAMPLE {i} OF {len(sample_indices)} {'*'*20}")
        print(f"Document ID: {doc_id}")
        print(f"Original filename: {db_metadata.get('original_filename', 'N/A')}")

        # Load the corresponding JSON file
        json_file_path = os.path.join(json_store_dir, f"{doc_id}.json")

        if os.path.exists(json_file_path):
            try:
                with open(json_file_path, "r") as f:
                    json_data = json.load(f)

                # UUID validation at the file level
                if "_id" in json_data:
                    stored_id = json_data.get("_id", "")
                    if stored_id and stored_id != doc_id:
                        print(
                            f"❌ UUID MISMATCH: Vector DB ID ({doc_id}) != JSON stored ID ({stored_id})"
                        )
                        content_validation_successful = False

                # Summary content validation
                json_summary = json_data.get("_summary_text", "")

                print("\nSummary in vector database:")
                print(f"    {db_summary}")
                print("\nSummary in JSON file:")
                print(f"    {json_summary}")

                # Check if summaries match
                if db_summary == json_summary:
                    print("\n✅ MATCH: Vector DB summary matches JSON file summary")
                else:
                    print(
                        "\n❌ MISMATCH: Vector DB summary DOES NOT match JSON file summary"
                    )
                    content_validation_successful = False

                # Print additional metadata from JSON for verification
                print("\nSelected JSON content for verification:")
                if "Demographics" in json_data:
                    if "Age" in json_data["Demographics"]:
                        print(f"    Age: {json_data['Demographics']['Age']}")
                    if "Sex" in json_data["Demographics"]:
                        print(f"    Sex: {json_data['Demographics']['Sex']}")

                if (
                    "Clinical Presentation" in json_data
                    and "Chief complaint" in json_data["Clinical Presentation"]
                ):
                    complaint = json_data["Clinical Presentation"]["Chief complaint"]
                    # Truncate if too long
                    if len(complaint) > 100:
                        complaint = complaint[:97] + "..."
                    print(f"    Chief complaint: {complaint}")

                # Compare metadata
                metadata_match = True
                for key in ["Age", "Sex", "Triage Category", "Diagnosis"]:
                    if key in db_metadata:
                        json_val = None
                        if key in ["Age", "Sex"] and "Demographics" in json_data:
                            json_val = json_data.get("Demographics", {}).get(key)
                        elif (
                            key in ["Triage Category", "Diagnosis"]
                            and "Case Disposition" in json_data
                        ):
                            json_val = json_data.get("Case Disposition", {}).get(key)

                        if json_val and db_metadata[key] != json_val:
                            print(
                                f"❌ METADATA MISMATCH for {key}: Vector DB ({db_metadata[key]}) != JSON ({json_val})"
                            )
                            metadata_match = False

                if metadata_match:
                    print("✅ Metadata values match between vector DB and JSON file")

            except Exception as e:
                print(f"\n❌ ERROR: Failed to load or parse JSON file: {str(e)}")
                content_validation_successful = False
        else:
            print(f"\n❌ ERROR: JSON file not found at {json_file_path}")
            content_validation_successful = False

    # Overall validation result
    overall_success = True
    if check_all_uuids:
        overall_success = uuid_validation_success and content_validation_successful
    else:
        overall_success = content_validation_successful

    print(f"\n{'='*30} VALIDATION SUMMARY {'='*30}")
    if check_all_uuids:
        print(
            f"UUID Validation: {'✅ PASSED' if uuid_validation_success else '❌ FAILED'}"
        )
    print(
        f"Content Validation: {'✅ PASSED' if content_validation_successful else '❌ FAILED'}"
    )
    print(f"Overall Validation: {'✅ PASSED' if overall_success else '❌ FAILED'}")

    return overall_success

## Create Vector Databases

This section loads the case data, processes it through the pipeline defined above, and creates the vector database. For each case:
1. Original JSON file is loaded
2. Metadata is extracted
3. An LLM-generated summary is created
4. The summary is embedded and stored in the vector database
5. The enriched JSON (original + summary) is saved to disk

This approach allows for efficient semantic search while maintaining access to the complete case data.

In [ ]:
# Define parameters
data_directory = "../data/past_case/json"
db_base_path = f"../db/past_case/db_{model}"
files_per_stage = 1000
max_files = 3000

# Create progressive vector databases
result = progressive_vector_db_creation(
    data_directory=data_directory,
    db_base_path=db_base_path,
    files_per_stage=files_per_stage,
    max_files=max_files,
)

print("\nProgressive Database Creation Summary:")
print(f"Total stages: {result['total_stages']}")
print(f"Total files processed: {result['total_files_processed']}")
print(f"Final database path: {result['final_database_path']}")

# Validate the final database
final_db_path = result["final_database_path"]
storage_dirs = create_directory_structure(final_db_path)
embeddings = initialize_embeddings()

# Load the final vector database
try:
    summary_vectordb = Chroma(
        embedding_function=embeddings,
        persist_directory=storage_dirs["summary_vectordb"],
        collection_name="summary_text",
    )

    # Validate with random samples
    validate_vector_database(
        vectordb=summary_vectordb,
        json_store_dir=storage_dirs["json_store"],
        num_samples=5,
    )
except Exception as e:
    print(f"Error loading or validating the final database: {str(e)}")

print("\nProcess completed successfully!")

In [ ]:
# Specify the database to validate
db_path = (
    f"../db/past_case/db_{model}_3000case"  # Change to the specific DB to validate
)

# Initialize embeddings
embeddings = initialize_embeddings()

# Set up directory paths
json_store_dir = os.path.join(db_path, "json_store")
summary_vectordb_path = os.path.join(db_path, "summary_vectordb")

# Load the vector database
summary_vectordb = Chroma(
    embedding_function=embeddings,
    persist_directory=summary_vectordb_path,
    collection_name="summary_text",
)

# Validate the database
validate_vector_database(
    vectordb=summary_vectordb,
    json_store_dir=json_store_dir,
    num_samples=5,
    check_all_uuids=True,
)